In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.tree


>DOM tree
output-file: core.dom.tree
title: core.dom.tree

This notebook provides utility functions for DOM tree manipulation
---

In [ ]:
# | default_exp core.dom.tree

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
from collections.abc import Callable
from typing import Optional

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
)


In [ ]:
# | hide
import asyncio
import threading
import io
from contextlib import redirect_stdout

In [ ]:
# | hide
def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, object] = {}
    error: dict[str, BaseException] = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

In [ ]:
# | export
def map_tree(node, transform: Callable):
    """
    Recursively apply a transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.

    Returns:
        The transformed tree.
    """
    
    node = transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    map_tree(child, transform)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = map_tree(value, transform)
    elif isinstance(node, list):
        node = [
            map_tree(child, transform) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node

In [ ]:
# | hide
def test_map_tree_applies_transform_recursively():
    tree = {
        "name": "root",
        "children": [
            {"name": "leaf"},
            [1, {"name": "nested"}],
        ],
    }

    def marker(node):
        if isinstance(node, dict):
            return {**node, "seen": True}
        if isinstance(node, list):
            return ["list-start", *node]
        return node

    result = map_tree(tree, marker)

    test_eq(result["seen"], True)
    test_eq(result["children"][0]["seen"], True)
    test_eq(result["children"][1][0], "list-start")
    test_eq(result["children"][1][2]["seen"], True)


test_map_tree_applies_transform_recursively()

In [ ]:
# | export
async def map_tree_async(node, transform: Callable, on_exit: Optional[Callable] = None):
    """Recursively apply an asynchronous transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.
        on_exit: An optional callable that is called with the node after it has been transformed.

    Returns:
        The transformed tree.
    """
    
    node = await transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    await map_tree_async(child, transform, on_exit)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = await map_tree_async(value, transform, on_exit)
    elif isinstance(node, list):
        node = [
            await map_tree_async(child, transform, on_exit) if isinstance(child, (dict, list)) else child
            for child in node
        ]

    if on_exit is not None:
        await on_exit(node)
    return node


In [ ]:
# | hide
def test_map_tree_async_applies_transform_and_on_exit():
    tree = {
        "name": "root",
        "children": [
            {"name": "leaf"},
            [1, {"name": "nested"}],
        ],
    }
    exited = []

    async def marker(node):
        if isinstance(node, dict):
            return {**node, "async_seen": True}
        if isinstance(node, list):
            return [*node, "list-end"]
        return node

    async def on_exit(node):
        exited.append(type(node).__name__)

    result = run_async(map_tree_async(tree, marker, on_exit=on_exit))

    test_eq(result["async_seen"], True)
    test_eq(result["children"][0]["async_seen"], True)
    test_eq(result["children"][1][-1], "list-end")
    test_eq(exited.count("dict") >= 3, True)
    test_eq(exited.count("list") >= 1, True)


test_map_tree_async_applies_transform_and_on_exit()

In [ ]:
# | export
def walk_node_obj(node, action:Optional[Callable]=None)->None:
    if action:
        node = action(node)
    if isinstance(node, ConfigJSONObject):
        # Dictionary
        for key, value in node:
            print(f"key   \"{key}\" at line {jsoncfg.node_location(value).line}")
            walk_node_obj(value)
    elif isinstance(node, ConfigJSONArray):
        # Array
        for item in node:
            walk_node_obj(item)
    elif isinstance(node, ConfigJSONScalar):
        # Scalar
        value = node()
        if isinstance(value, str):
            value = value.strip()
        print(f"value \"{value}\" at line {jsoncfg.node_location(node).line}")
    else:
        raise ValueError(f"Unknown node type: {type(node)}")

In [ ]:
# | hide
def test_walk_node_obj_prints_locations_and_rejects_unknown_nodes():
    config = jsoncfg.loads_config('{"title": " Hello ", "items": [1, "x"]}')
    stdout = io.StringIO()
    with redirect_stdout(stdout):
        walk_node_obj(config)

    output = stdout.getvalue()
    test_eq('key   "title"' in output, True)
    test_eq('value "Hello"' in output, True)
    test_eq('value "1"' in output, True)
    test_fail(lambda: walk_node_obj(123), contains='Unknown node type')


test_walk_node_obj_prints_locations_and_rejects_unknown_nodes()

In [ ]:
# | export
def walk_nodes_with_line_number(raw_json: str, action: Optional[Callable] = None) -> None:
    """Walk the AST config structure and print node locations by source line.
    
    Args:
        action: Optional transform applied to each visited node.
    """
    if action is None:
        def _identity(node):
            return node
        action = _identity

    ast = jsoncfg.loads_config(raw_json)


    ast = walk_node_obj(ast)


In [ ]:
# | hide
def test_walk_nodes_with_line_number_prints_walk_output():
    raw_json = '''
    {
      "title": "Hello",
      "items": [1, "x"]
    }
    '''
    stdout = io.StringIO()
    with redirect_stdout(stdout):
        result = walk_nodes_with_line_number(raw_json)

    output = stdout.getvalue()
    test_eq(result, None)
    test_eq('key   "title"' in output, True)
    test_eq('value "Hello"' in output, True)
    test_eq('line' in output, True)


test_walk_nodes_with_line_number_prints_walk_output()

In [ ]:
# | export
def walk_node(node, action: Optional[Callable] = None):
    if action:
        node = action(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    walk_node(child)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = walk_node(value)
    elif isinstance(node, list):
        node = [
            walk_node(child, action) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node

In [ ]:
# | hide
def test_walk_node_walks_nested_lists_and_top_level_action():
    tree = [
        {"name": "root", "children": [{"name": "leaf"}]},
        {"name": "second"},
    ]

    def marker(node):
        if isinstance(node, dict):
            return {**node, "visited": True}
        return node

    result = walk_node(tree, marker)

    test_eq(result[0]["visited"], True)
    test_eq(result[0]["children"][0]["name"], "leaf")
    test_eq(result[1]["visited"], True)


test_walk_node_walks_nested_lists_and_top_level_action()

In [ ]:
# | export
def walk(raw_json:str, action: Optional[Callable] = None) -> str:
        """ Walks through the Markdown AST and applies an action to each node.
        If no action is provided, it defaults to the identity function.

        returns:
            str: The modified JSON string after applying the action to each node.
        """

        ast = json.loads(raw_json)
        ast = walk_node(ast, action)
        return json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
# | hide
def test_walk_returns_json_string_for_transformed_tree():
    raw_json = json.dumps([
        {"name": "root"},
        {"name": "leaf"},
    ])

    def marker(node):
        if isinstance(node, dict):
            return {**node, "done": True}
        return node

    result = json.loads(walk(raw_json, marker))

    test_eq(result[0]["done"], True)
    test_eq(result[1]["done"], True)
    test_eq(result[0]["name"], "root")


test_walk_returns_json_string_for_transformed_tree()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()